In [ ]:
%%capture
!pip install unsloth[colab-new] -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install datasets huggingface_hub -q

In [ ]:
import os, json, torch
from unsloth import FastLanguageModel
from trl import SFTTrainer, TrainingArguments, DataCollatorForSeq2Seq
from datasets import load_dataset, Dataset
from huggingface_hub import login

# Kaggle > Add-ons > Secrets > HF_TOKEN ekle
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN)

MODEL_NAME  = 'Qwen/Qwen2.5-7B-Instruct'
OUTPUT_NAME = 'mironlaw-1.0'
HF_REPO     = 'mironintelligence/MironLaw-1.0'

MAX_SEQ_LEN  = 2048
LORA_RANK    = 64
BATCH_SIZE   = 2
GRAD_ACCUM   = 8
LR           = 2e-4
EPOCHS       = 2
WARMUP_STEPS = 100

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    use_rslora=True,
)
print(model.print_trainable_parameters())

In [ ]:
def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

try:
    ds = load_dataset('mironintelligence/mironlaw-train-data', split='train')
    print(f'HF dataset yüklendi: {len(ds):,} örnek')
except Exception as e:
    print(f'HF dataset yüklenemedi ({e}), yerel dosya aranıyor...')
    data = load_jsonl('/kaggle/input/mironlaw-data/train.jsonl')
    ds = Dataset.from_list(data)
    print(f'Yerel dosyadan yüklendi: {len(ds):,} örnek')

def apply_chat_template(examples):
    texts = []
    for messages in examples['messages']:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )
        texts.append(text)
    return {'text': texts}

ds = ds.map(apply_chat_template, batched=True)
print(f'Dataset hazır: {len(ds):,} örnek')
print('Örnek:\n', ds[0]['text'][:500])

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=50,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir=OUTPUT_NAME,
        report_to='none',
        save_strategy='steps',
        save_steps=500,
        save_total_limit=3,
    ),
)

print('Training başlıyor...')
trainer_stats = trainer.train()
print(f'Training tamamlandı! Loss: {trainer_stats.training_loss:.4f}')
print(f'Süre: {trainer_stats.metrics["train_runtime"]:.0f}s')

In [ ]:
# Inference test
FastLanguageModel.for_inference(model)

test_messages = [
    {'role': 'system', 'content': 'Sen MironLaw 1.0, Türk hukuku uzmanı bir yapay zeka asistanısın.'},
    {'role': 'user', 'content': 'İş kazasında işverenin hukuki sorumluluğu nedir?'},
]

inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors='pt',
).to('cuda')

outputs = model.generate(
    input_ids=inputs, max_new_tokens=512,
    temperature=0.3, top_p=0.9, do_sample=True,
)

print('MironLaw 1.0 Yanıtı:')
print('=' * 60)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# Merged model + GGUF olarak HF'e push et
model.push_to_hub_merged(
    HF_REPO, tokenizer,
    save_method='merged_16bit',
    token=HF_TOKEN,
)

model.push_to_hub_gguf(
    HF_REPO + '-GGUF', tokenizer,
    quantization_method='q4_k_m',
    token=HF_TOKEN,
)

print(f'Model: https://huggingface.co/{HF_REPO}')
print(f'GGUF:  https://huggingface.co/{HF_REPO}-GGUF')